# Unidad 4: Persistencia y Bases de Datos
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## Introducción y Contexto de Negocio

En cualquier negocio digital, la **información es el activo más valioso**. Un e-commerce necesita registrar productos, stock, clientes e historial de pedidos. Una fintech requiere registrar transacciones con precisión atómica y absoluta consistencia.

Para lograr esto, necesitamos **Persistencia de Datos**. En esta unidad abordaremos la interacción de Python con bases de datos estructuradas relacionales:
1. **SQL Directo**: Consultas e interacciones directas con el motor ligero **SQLite** integrado en Python.
2. **Mapeo Objeto-Relacional (ORM)**: El estándar de la industria que traduce tablas relacionales en clases de Python y registros en objetos. Utilizaremos el ORM líder de la industria: **SQLAlchemy**.

### Objetivos de Aprendizaje:
1. Comprender la arquitectura de almacenamiento de datos relacional.
2. Conectarse a motores SQL y ejecutar operaciones CRUD mediante comandos directos de SQL en Python.
3. Comprender los conceptos de un ORM y mapear bases de datos relacionales a código estructurado.
4. Construir modelos relacionales avanzados (Relaciones 1 a N, claves foráneas) con SQLAlchemy.


## 1. Conexión SQL Directa con `sqlite3`

SQLite es un motor de base de datos relacional integrado en la biblioteca estándar de Python. Es excelente para el desarrollo ágil de prototipos porque almacena toda la base de datos en un solo archivo físico local.


In [2]:
import sqlite3

# Conectar a la base de datos (se crea el archivo 'tienda.db' si no existe)
conexion = sqlite3.connect("tienda.db")
cursor = conexion.cursor()

# 1. Crear una tabla de productos
cursor.execute("""
CREATE TABLE IF NOT EXISTS productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL,
    stock INTEGER NOT NULL
)
""")
conexion.commit()
print("Tabla 'productos' creada exitosamente.")


Tabla 'productos' creada exitosamente.


### Operaciones CRUD (Create, Read, Update, Delete) directas en SQL


In [3]:
# 2. CREATE: Insertar registros
productos_iniciales = [
    ("Licencia CRM", 120.0, 50),
    ("Módulo de Leads", 45.0, 100),
    ("API Gateway Pro", 250.0, 15)
]

cursor.executemany("INSERT INTO productos (nombre, precio, stock) VALUES (?, ?, ?)", productos_iniciales)
conexion.commit()
print(f"Registros insertados: {cursor.rowcount}")

# 3. READ: Consultar registros
cursor.execute("SELECT * FROM productos WHERE precio > 50.0")
filas = cursor.fetchall()
print("\nProductos con precio mayor a $50:")
for fila in filas:
    print(f"ID: {fila[0]} | Nombre: {fila[1]} | Precio: ${fila[2]} | Stock: {fila[3]}")

# 4. UPDATE: Actualizar stock de un producto
cursor.execute("UPDATE productos SET stock = stock - 1 WHERE nombre = ?", ("Licencia CRM",))
conexion.commit()

# 5. DELETE: Eliminar producto con stock cero o bajo (ejemplo simulado)
cursor.execute("DELETE FROM productos WHERE stock = 0")
conexion.commit()

# Cerramos el cursor y conexión al finalizar
cursor.close()
conexion.close()


Registros insertados: 3

Productos con precio mayor a $50:
ID: 1 | Nombre: Licencia CRM | Precio: $120.0 | Stock: 50
ID: 3 | Nombre: API Gateway Pro | Precio: $250.0 | Stock: 15


### Consultas Avanzadas: Relacionando Tablas mediante `JOIN` en SQL Embebido

Para consultar información distribuida en tablas relacionales utilizando SQL nativo, ejecutamos la cláusula `INNER JOIN` o `LEFT JOIN` pasando la clave foránea como condición de enlace.

In [5]:
import sqlite3

# 1. Reabrir o crear la conexión y el cursor
conexion = sqlite3.connect("tienda.db")
cursor = conexion.cursor()

# 2. Crear las tablas relacionales requeridas si aún no existen
cursor.execute("""
CREATE TABLE IF NOT EXISTS clientes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS pedidos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    monto REAL NOT NULL,
    cliente_id INTEGER,
    FOREIGN KEY (cliente_id) REFERENCES clientes(id)
);
""")
conexion.commit()

# 3. Cargar datos de prueba si las tablas están vacías
cursor.execute("SELECT COUNT(*) FROM clientes")
if cursor.fetchone()[0] == 0:
    cursor.execute("INSERT INTO clientes (nombre, email) VALUES ('Sofía Rivas', 'sofia@umsa.edu.ar')")
    cursor.execute("INSERT INTO clientes (nombre, email) VALUES ('Martín Gómez', 'martin@umsa.edu.ar')")
    cursor.execute("INSERT INTO pedidos (monto, cliente_id) VALUES (350.00, 1)")
    cursor.execute("INSERT INTO pedidos (monto, cliente_id) VALUES (120.50, 1)")
    cursor.execute("INSERT INTO pedidos (monto, cliente_id) VALUES (500.00, 2)")
    conexion.commit()

# 4. Ejemplo de consulta INNER JOIN combinando Clientes y Pedidos en SQL puro
cursor.execute("""
SELECT c.nombre, c.email, p.id AS pedido_id, p.monto
FROM clientes c
INNER JOIN pedidos p ON c.id = p.cliente_id;
""")

filas_join = cursor.fetchall()

print("=== Resultado de SQL Embebido con INNER JOIN ===")
for fila in filas_join:
    print(f"Cliente: {fila[0]} ({fila[1]}) | Pedido #{fila[2]} | Monto: ${fila[3]:.2f}")

# 5. Cerrar la conexión únicamente al finalizar todas las consultas
cursor.close()
conexion.close()

=== Resultado de SQL Embebido con INNER JOIN ===
Cliente: Sofía Rivas (sofia@umsa.edu.ar) | Pedido #1 | Monto: $350.00
Cliente: Sofía Rivas (sofia@umsa.edu.ar) | Pedido #2 | Monto: $120.50
Cliente: Martín Gómez (martin@umsa.edu.ar) | Pedido #3 | Monto: $500.00


## 2. Introducción al Mapeo Objeto-Relacional (ORM) con SQLAlchemy

Escribir consultas SQL en bruto como strings dentro de Python puede ser propenso a errores de tipado e inyección de código. Un **ORM (Object-Relational Mapping)** nos permite manipular la base de datos usando clases y objetos nativos de Python.

Utilizaremos **SQLAlchemy** (declarative style) para crear nuestros modelos.


In [6]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

# Crear motor de base de datos en memoria para este ejemplo interactivo
engine = create_engine("sqlite:///:memory:", echo=False) # echo=True muestra el SQL generado
Base = declarative_base()

# Definición del Modelo (Mapea la clase a la tabla 'clientes')
class Cliente(Base):
    __tablename__ = 'clientes'

    id = Column(Integer, primary_key=True)
    nombre = Column(String(50), nullable=False)
    email = Column(String(50), unique=True, nullable=False)

    # Relación 1-a-N con Pedido
    pedidos = relationship("Pedido", back_populates="cliente", cascade="all, delete-orphan")

# Definición del Modelo de Pedidos
class Pedido(Base):
    __tablename__ = 'pedidos'

    id = Column(Integer, primary_key=True)
    monto = Column(Float, nullable=False)
    cliente_id = Column(Integer, ForeignKey('clientes.id'))

    cliente = relationship("Cliente", back_populates="pedidos")

# Crear las tablas físicamente en la base de datos
Base.metadata.create_all(engine)
print("Tablas 'clientes' y 'pedidos' generadas mediante SQLAlchemy.")


Tablas 'clientes' y 'pedidos' generadas mediante SQLAlchemy.


## 3. Persistencia NoSQL para Analítica Web y Documentos JSON (MongoDB / `pymongo`)

A diferencia del modelo relacional rígido (tablas, filas y columnas), las bases de datos **NoSQL orientadas a documentos** (como MongoDB) almacenan la información en estructuras flexibles similares a JSON (llamadas BSON).

Son el estándar de la industria para:
- Registrar eventos de navegación en tiempo real (*clickstream*).
- Almacenar payloads de APIs con estructuras dinámicas o cambiantes.
- Logs de auditoría y métricas analíticas de e-commerce.

In [7]:
# Ejemplo de integración con MongoDB usando la librería oficial pymongo
import json

# Simulación de un log de analítica web en formato JSON / Diccionario
evento_clickstream = {
    "sesion_id": "sess_883921",
    "usuario_id": 1024,
    "evento": "producto_agregado_al_carrito",
    "metadata": {
        "producto_id": "PROD-99",
        "categoria": "SaaS",
        "precio": 120.00,
        "dispositivo": "Mobile iOS"
    },
    "timestamp": "2026-08-06T10:15:00Z"
}

# Conexión e inserción con pymongo (Ejemplo de sintaxis estándar)
"""
import pymongo

# Conexión al servidor de MongoDB
cliente_mongo = pymongo.MongoClient("mongodb://localhost:27017/")
db = cliente_mongo["analytics_ecommerce"]
coleccion_eventos = db["logs_eventos"]

# INSERT: Insertar un documento JSON
resultado = coleccion_eventos.insert_one(evento_clickstream)
print(f"Documento NoSQL insertado con ID: {resultado.inserted_id}")

# READ: Consultar documentos por atributo dinámico
eventos_mobile = coleccion_eventos.find({"metadata.dispositivo": "Mobile iOS"})
for ev in eventos_mobile:
    print(ev)
"""

print("Estructura del Documento JSON listo para persistir en MongoDB:")
print(json.dumps(evento_clickstream, indent=2, ensure_ascii=False))

Estructura del Documento JSON listo para persistir en MongoDB:
{
  "sesion_id": "sess_883921",
  "usuario_id": 1024,
  "evento": "producto_agregado_al_carrito",
  "metadata": {
    "producto_id": "PROD-99",
    "categoria": "SaaS",
    "precio": 120.0,
    "dispositivo": "Mobile iOS"
  },
  "timestamp": "2026-08-06T10:15:00Z"
}


### Ejecución de operaciones mediante la Sesión de SQLAlchemy


In [8]:
# Configurar la fábrica de sesiones
Session = sessionmaker(bind=engine)
sesion = Session()

# 1. CREATE: Insertar un cliente con un pedido
cliente_nuevo = Cliente(nombre="Sofía Rivas", email="sofia@umsa.edu.ar")
pedido_1 = Pedido(monto=350.0, cliente=cliente_nuevo)
pedido_2 = Pedido(monto=120.5, cliente=cliente_nuevo)

sesion.add(cliente_nuevo)
sesion.commit() # Guarda de forma atómica en la BD
print("Cliente y pedidos guardados con éxito.")

# 2. READ: Consultar registros a través de objetos de Python
cliente_db = sesion.query(Cliente).filter_by(email="sofia@umsa.edu.ar").first()
print(f"\nCliente encontrado: {cliente_db.nombre}")
print(f"Pedidos asociados a {cliente_db.nombre}:")
for ped in cliente_db.pedidos:
    print(f" -> Pedido ID: {ped.id} | Monto: ${ped.monto}")

# 3. UPDATE: Modificar el monto del pedido
pedido_db = sesion.query(Pedido).filter_by(id=1).first()
pedido_db.monto = 399.99
sesion.commit()
print(f"\nMonto del Pedido 1 modificado a: ${pedido_db.monto}")

# Cerramos la sesión
sesion.close()


Cliente y pedidos guardados con éxito.

Cliente encontrado: Sofía Rivas
Pedidos asociados a Sofía Rivas:
 -> Pedido ID: 1 | Monto: $350.0
 -> Pedido ID: 2 | Monto: $120.5

Monto del Pedido 1 modificado a: $399.99


---

## Desafío Práctico (Trabajo Práctico 4)

**Consigna de Negocio (Base de Datos de Suscripciones SaaS):**
Debes diseñar e implementar un backend relacional para una plataforma SaaS.

1. Define un modelo SQLAlchemy para `UsuarioSaaS` con campos:
   - `id` (entero, clave primaria)
   - `nombre` (string)
   - `email` (string, único)
2. Define un modelo `SuscripcionSaaS` con campos:
   - `id` (entero, clave primaria)
   - `plan` (string, ej. `"Basic"`, `"Enterprise"`)
   - `precio_mensual` (float)
   - `usuario_id` (entero, clave foránea vinculando a `UsuarioSaaS`)
3. Establece la relación de uno a muchos (o uno a uno, según lo consideres) indicando la vinculación entre ambos modelos.
4. Inicializa un motor SQLite en memoria y crea las tablas correspondientes.
5. Inicia una sesión e implementa las siguientes operaciones de prueba:
   - Inserta al menos 2 usuarios.
   - Asigna una suscripción de plan `"Enterprise"` ($199.99/mes) al usuario 1 y una suscripción de plan `"Basic"` ($29.99/mes) al usuario 2.
   - Ejecuta una consulta agregada que devuelva el ingreso mensual total de la plataforma (la suma de los precios mensuales de todas las suscripciones activas).
   - Modifica la suscripción del usuario 2 para actualizar su plan a `"Pro"` ($79.99/mes) y vuelve a consultar el ingreso total final para verificar la actualización de la persistencia.

Implementa tu solución a continuación.


In [ ]:
# Escribe la resolución aquí
# 1. Definir Modelos
# ...

# 2. Configurar Base de Datos e insertar registros
# ...

# 3. Consultas e ingresos totales
# ...


In [9]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, func
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

# 1. Definición de la Base de SQLAlchemy
Base = declarative_base()

# 2. Modelo UsuarioSaaS
class UsuarioSaaS(Base):
    __tablename__ = 'usuarios_saas'

    id = Column(Integer, primary_key=True)
    nombre = Column(String(100), nullable=False)
    email = Column(String(100), unique=True, nullable=False)

    # Relación 1 a N (un usuario puede tener suscripciones)
    suscripciones = relationship("SuscripcionSaaS", back_populates="usuario", cascade="all, delete-orphan")

    def __repr__(self):
        return f"<UsuarioSaaS(id={self.id}, nombre='{self.nombre}', email='{self.email}')>"

# 3. Modelo SuscripcionSaaS
class SuscripcionSaaS(Base):
    __tablename__ = 'suscripciones_saas'

    id = Column(Integer, primary_key=True)
    plan = Column(String(50), nullable=False)
    precio_mensual = Column(Float, nullable=False)
    usuario_id = Column(Integer, ForeignKey('usuarios_saas.id'), nullable=False)

    # Relación inversa
    usuario = relationship("UsuarioSaaS", back_populates="suscripciones")

    def __repr__(self):
        return f"<SuscripcionSaaS(id={self.id}, plan='{self.plan}', precio_mensual=${self.precio_mensual:.2f})>"

# 4. Inicializar motor SQLite en memoria y crear tablas
engine = create_engine("sqlite:///:memory:", echo=False)
Base.metadata.create_all(engine)

# 5. Configuración e inicio de Sesión
Session = sessionmaker(bind=engine)
session = Session()

try:
    print("=== INICIO DE OPERACIONES DE PRUEBA ===")

    # a. Insertar al menos 2 usuarios
    usuario_1 = UsuarioSaaS(nombre="Ana Gómez", email="ana.gomez@empresa.com")
    usuario_2 = UsuarioSaaS(nombre="Carlos Pérez", email="carlos.perez@startup.io")

    session.add_all([usuario_1, usuario_2])
    session.commit() # Guardamos para obtener los IDs generados

    # b. Asignar suscripción Enterprise a usuario 1 y Basic a usuario 2
    sub_1 = SuscripcionSaaS(plan="Enterprise", precio_mensual=199.99, usuario=usuario_1)
    sub_2 = SuscripcionSaaS(plan="Basic", precio_mensual=29.99, usuario=usuario_2)

    session.add_all([sub_1, sub_2])
    session.commit()
    print("Users y Suscripciones iniciales insertadas con éxito.")

    # c. Consulta agregada: Ingreso mensual total de la plataforma
    ingreso_total_inicial = session.query(func.sum(SuscripcionSaaS.precio_mensual)).scalar() or 0.0
    print(f"\n[CONSULTA AGREGADA] Ingreso Mensual Total Inicial: ${ingreso_total_inicial:.2f}")

    # d. Modificar la suscripción del usuario 2 a "Pro" ($79.99/mes)
    sub_usuario_2 = session.query(SuscripcionSaaS).filter_by(usuario_id=usuario_2.id).first()
    if sub_usuario_2:
        sub_usuario_2.plan = "Pro"
        sub_usuario_2.precio_mensual = 79.99
        session.commit()
        print(f"\n[ACTUALIZACIÓN] Plan de {usuario_2.nombre} actualizado a 'Pro' ($79.99/mes).")

    # e. Volver a consultar el ingreso total final para verificar la persistencia
    ingreso_total_final = session.query(func.sum(SuscripcionSaaS.precio_mensual)).scalar() or 0.0
    print(f"[CONSULTA AGREGADA] Ingreso Mensual Total Actualizado: ${ingreso_total_final:.2f}")

    print("\n=== FIN DE OPERACIONES ===")

finally:
    # Cerrar la sesión
    session.close()

=== INICIO DE OPERACIONES DE PRUEBA ===
Users y Suscripciones iniciales insertadas con éxito.

[CONSULTA AGREGADA] Ingreso Mensual Total Inicial: $229.98

[ACTUALIZACIÓN] Plan de Carlos Pérez actualizado a 'Pro' ($79.99/mes).
[CONSULTA AGREGADA] Ingreso Mensual Total Actualizado: $279.98

=== FIN DE OPERACIONES ===
